# 03. Training SVM

Notebook ini melatih model Support Vector Machine pada dataset MNIST dan Fashion MNIST menggunakan empat kernel:

1. Linear
2. Polynomial degree 3
3. RBF
4. Sigmoid

SVM menggunakan subset data tetap agar komputasi tetap aman. Hasil evaluasi disimpan ke `results/svm_results.csv`.

In [1]:
import os
import sys
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import Subset

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
def get_project_root():
    cwd = Path.cwd().resolve()
    if cwd.name.lower() == "notebooks":
        return cwd.parent
    return cwd

PROJECT_ROOT = get_project_root()

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
CM_DIR = RESULTS_DIR / "confusion_matrix"

for path in [DATA_DIR, RESULTS_DIR, CM_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: C:\UTS_Data_Mining


In [3]:
CONFIG = {
    "random_seed": 42,
    "svm_train_subset": 10000,
    "svm_test_subset": 2000,
    "input_size": 784,
    "num_classes": 10
}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(CONFIG["random_seed"])
print("Random seed:", CONFIG["random_seed"])

Random seed: 42


In [4]:
transform = transforms.ToTensor()

dataset_objects = {
    "MNIST": {
        "train": datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=transform),
        "test": datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=transform)
    },
    "FashionMNIST": {
        "train": datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=transform),
        "test": datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=transform)
    }
}

for name, obj in dataset_objects.items():
    print(f"{name}: train={len(obj['train'])}, test={len(obj['test'])}")

MNIST: train=60000, test=10000
FashionMNIST: train=60000, test=10000


In [5]:
def fixed_indices(dataset_length, subset_size, seed=42):
    subset_size = min(subset_size, dataset_length)
    rng = np.random.RandomState(seed)
    return rng.choice(dataset_length, size=subset_size, replace=False)


def dataset_to_numpy(dataset, indices):
    X = []
    y = []

    for idx in indices:
        image, label = dataset[idx]
        X.append(image.view(-1).numpy())
        y.append(label)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)

    return X, y


def prepare_svm_data(train_dataset, test_dataset):
    train_indices = fixed_indices(
        len(train_dataset),
        CONFIG["svm_train_subset"],
        seed=CONFIG["random_seed"]
    )

    test_indices = fixed_indices(
        len(test_dataset),
        CONFIG["svm_test_subset"],
        seed=CONFIG["random_seed"]
    )

    X_train, y_train = dataset_to_numpy(train_dataset, train_indices)
    X_test, y_test = dataset_to_numpy(test_dataset, test_indices)

    return X_train, y_train, X_test, y_test

In [6]:
def calculate_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_score": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }


svm_configs = {
    "SVM Linear": {
        "kernel": "linear",
        "configuration": "StandardScaler + SVC(kernel='linear', C=1.0)"
    },
    "SVM Polynomial": {
        "kernel": "poly",
        "configuration": "StandardScaler + SVC(kernel='poly', degree=3, C=1.0, gamma='scale')"
    },
    "SVM RBF": {
        "kernel": "rbf",
        "configuration": "StandardScaler + SVC(kernel='rbf', C=1.0, gamma='scale')"
    },
    "SVM Sigmoid": {
        "kernel": "sigmoid",
        "configuration": "StandardScaler + SVC(kernel='sigmoid', C=1.0, gamma='scale')"
    }
}


def build_svm_model(kernel):
    if kernel == "poly":
        svm = SVC(kernel=kernel, degree=3, C=1.0, gamma="scale")
    else:
        svm = SVC(kernel=kernel, C=1.0, gamma="scale")

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", svm)
    ])

    return model

In [7]:
results = []

for dataset_name, dataset_data in dataset_objects.items():
    print("=" * 80)
    print(f"Dataset: {dataset_name}")

    X_train, y_train, X_test, y_test = prepare_svm_data(
        dataset_data["train"],
        dataset_data["test"]
    )

    print("Shape X_train:", X_train.shape)
    print("Shape X_test :", X_test.shape)

    for model_name, config in svm_configs.items():
        print("-" * 80)
        print(f"Training {model_name} pada {dataset_name}")

        start_time = time.time()

        svm_model = build_svm_model(config["kernel"])
        svm_model.fit(X_train, y_train)

        training_time = time.time() - start_time

        y_pred = svm_model.predict(X_test)
        metrics = calculate_metrics(y_test, y_pred)

        row = {
            "dataset": dataset_name,
            "model_group": "SVM",
            "model": model_name,
            "configuration": config["configuration"],
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1_score": metrics["f1_score"],
            "training_time_seconds": training_time,
            "train_size": len(y_train),
            "test_size": len(y_test)
        }

        results.append(row)

        print("Hasil evaluasi:")
        print(row)

Dataset: MNIST
Shape X_train: (10000, 784)
Shape X_test : (2000, 784)
--------------------------------------------------------------------------------
Training SVM Linear pada MNIST
Hasil evaluasi:
{'dataset': 'MNIST', 'model_group': 'SVM', 'model': 'SVM Linear', 'configuration': "StandardScaler + SVC(kernel='linear', C=1.0)", 'accuracy': 0.9175, 'precision': 0.9177106879143615, 'recall': 0.9155824877148303, 'f1_score': 0.9161758992145573, 'training_time_seconds': 4.11592960357666, 'train_size': 10000, 'test_size': 2000}
--------------------------------------------------------------------------------
Training SVM Polynomial pada MNIST
Hasil evaluasi:
{'dataset': 'MNIST', 'model_group': 'SVM', 'model': 'SVM Polynomial', 'configuration': "StandardScaler + SVC(kernel='poly', degree=3, C=1.0, gamma='scale')", 'accuracy': 0.906, 'precision': 0.9163628710357109, 'recall': 0.9054642793574723, 'f1_score': 0.9074561317685781, 'training_time_seconds': 20.36637830734253, 'train_size': 10000, 'tes

In [8]:
svm_results_df = pd.DataFrame(results)
svm_results_df = svm_results_df.sort_values(
    by=["dataset", "f1_score"],
    ascending=[True, False]
).reset_index(drop=True)

output_path = RESULTS_DIR / "svm_results.csv"
svm_results_df.to_csv(output_path, index=False)

display(svm_results_df)
print("Hasil SVM disimpan ke:", output_path)

,dataset,model_group,model,configuration,accuracy,precision,recall,f1_score,training_time_seconds,train_size,test_size
0,FashionMNIST,SVM,SVM RBF,"StandardScaler + SVC(kernel='rbf', C=1.0, gamm...",0.8585,0.855469,0.858164,0.855729,8.501462,10000,2000
1,FashionMNIST,SVM,SVM Polynomial,"StandardScaler + SVC(kernel='poly', degree=3, ...",0.8260,0.845570,0.825543,0.831030,10.583226,10000,2000
2,FashionMNIST,SVM,SVM Linear,"StandardScaler + SVC(kernel='linear', C=1.0)",0.8015,0.801504,0.800477,0.799463,8.090992,10000,2000
3,FashionMNIST,SVM,SVM Sigmoid,"StandardScaler + SVC(kernel='sigmoid', C=1.0, ...",0.7405,0.737254,0.740814,0.736458,5.915385,10000,2000
4,MNIST,SVM,SVM RBF,"StandardScaler + SVC(kernel='rbf', C=1.0, gamm...",0.9375,0.938680,0.936547,0.936879,10.337077,10000,2000
5,MNIST,SVM,SVM Linear,"StandardScaler + SVC(kernel='linear', C=1.0)",0.9175,0.917711,0.915582,0.916176,4.115930,10000,2000
6,MNIST,SVM,SVM Polynomial,"StandardScaler + SVC(kernel='poly', degree=3, ...",0.9060,0.916363,0.905464,0.907456,20.366378,10000,2000
7,MNIST,SVM,SVM Sigmoid,"StandardScaler + SVC(kernel='sigmoid', C=1.0, ...",0.9065,0.906112,0.904871,0.905135,5.884655,10000,2000


Hasil SVM disimpan ke: C:\UTS_Data_Mining\results\svm_results.csv


## Catatan

SVM menggunakan subset data karena algoritma SVM, terutama kernel non-linear, dapat memerlukan waktu komputasi yang besar pada dataset citra berukuran besar.